# Structure-Aware + Fact-Consistent Scientific Summarization
## Phase 1 + Phase 2 (Figures/Tables Segmentation)

This notebook demonstrates a full research prototype addressing:
- Factual inconsistency
- Weak structure understanding
- Weak evaluation coverage
- Low controllability
- Missing domain intelligence
- Missing figures/tables integration (Phase 2)

## 1. Setup

In [1]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from research_experiment_framework import ResearchPipeline, pretty_metric_table

ROOT = Path.cwd()
FIG_DIR = ROOT / 'outputs' / 'figures'
TAB_DIR = ROOT / 'outputs' / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATHS = [str(ROOT / 'data' / '2004.05150v2.pdf')]
MODEL_PATH = ROOT / 'models' / 'llama-3.2-1b-instruct.Q4_K_M.gguf'
os.environ['SUMMARIX_MODEL_PATH'] = str(MODEL_PATH)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model file not found: {MODEL_PATH}')

pipeline = ResearchPipeline(use_local_model=True)
if not pipeline.llm.has_local_model():
    raise RuntimeError(
        'Local llama.cpp model did not load. Install llama-cpp-python in this environment and rerun. '
        f'Model path: {MODEL_PATH}'
    )

papers = pipeline.load_papers(PDF_PATHS, include_media=True)
print('Model status:', pipeline.llm.status_label())
print('Loaded papers:', len(papers))
print('Title:', papers[0]['metadata'].get('title'))
print('Authors:', papers[0]['metadata'].get('authors'))
print('Year:', papers[0]['metadata'].get('year'))
print('Sections:', len(papers[0]['sections']))
print('Figures detected:', len(papers[0]['figures']))
print('Tables detected:', len(papers[0]['tables']))


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Model status: Local model loaded
Loaded papers: 1
Title: Longformer: The Long-Document Transformer
Authors: Iz Beltagy∗, Matthew E. Peters∗, Arman Cohan∗
Year: 2020
Sections: 17
Figures detected: 4
Tables detected: 0


## 2. IDEA 1 + IDEA 2 + IDEA 4
Baseline vs Structure-aware vs Fact-checked evaluation

In [2]:
single_results = pipeline.run_single_paper_experiment(papers[0])
print(pretty_metric_table(single_results))
print('\nDetected domain:', single_results['detected_domain'])
print('Baseline selected sections:', single_results['baseline_selected_sections'])
print('Structure selected sections:', single_results['structure_selected_sections'])


Metric            | Baseline | Structure-aware | Fact-checked
------------------+----------+-----------------+-------------
rouge1_f1         | 0.1277   | 0.1346          | 0.1346      
rouge2_f1         | 0.0483   | 0.0957          | 0.0957      
rougeL_f1         | 0.0747   | 0.0832          | 0.0832      
semantic_f1_proxy | 0.4532   | 0.565           | 0.565       
factual_score     | 0.3235   | 0.5022          | 0.5022      
contradictions    | 0        | 0               | 0           
section_coverage  | 0.6      | 0.8             | 0.8         
graph_coherence   | 0.0      | 0.1663          | 0.1663      
structure_signal  | 0.0      | 0.1663          | 0.1663      
runtime_sec       | 151.459  | 237.094         | -           

Detected domain: legal
Baseline selected sections: ['Results', 'Abstract', 'Results (2)', 'Conclusion and Future Work']
Structure selected sections: ['Abstract', 'Conclusion and Future Work', 'Results', 'Results (2)', 'Task speciﬁc model details', 'Autore

## 3. Phase 2: Figures/Tables Segmentation + Metrics

In [3]:
phase2 = pipeline.run_phase2_media_experiment(papers[0])
print('Baseline media score:', phase2['baseline']['phase2_media_score'])
print('Proposed media score:', phase2['proposed']['phase2_media_score'])
print('Improvement:', phase2['improvement'].get('phase2_media_score'))
print('\nProposed media metrics:')
for k, v in phase2['proposed'].items():
    if k == 'per_item':
        continue
    print(f'- {k}: {v}')
print('\nSample media-segmentation assignments:')
for row in phase2['proposed']['per_item'][:8]:
    print(row)


Baseline media score: 0.0
Proposed media score: 0.4875
Improvement: 0.4875

Proposed media metrics:
- figure_count: 4
- table_count: 0
- media_total: 4
- figure_coverage: 1.0
- table_coverage: 0.0
- figure_caption_quality: 1.0
- table_preview_quality: 0.0
- figure_alignment: 0.5
- table_alignment: 0.0
- media_in_core_sections: 0.0
- media_density_per_page: 0.2353
- phase2_media_score: 0.4875

Sample media-segmentation assignments:
{'id': 'figure_3_1', 'type': 'figure', 'page': 3, 'assigned_sections': ['Related Work', 'Longformer'], 'description': 'Image extracted from page 3'}
{'id': 'figure_3_2', 'type': 'figure', 'page': 3, 'assigned_sections': ['Related Work', 'Longformer'], 'description': 'Image extracted from page 3'}
{'id': 'figure_3_3', 'type': 'figure', 'page': 3, 'assigned_sections': ['Related Work', 'Longformer'], 'description': 'Image extracted from page 3'}
{'id': 'figure_3_4', 'type': 'figure', 'page': 3, 'assigned_sections': ['Related Work', 'Longformer'], 'description': 

## 3A. Publication Plots and Tables

In [4]:
# PUB_PLOTS_TABLES
plt.style.use('seaborn-v0_8-whitegrid')

metric_keys = [
    'rouge1_f1', 'rouge2_f1', 'rougeL_f1', 'semantic_f1_proxy',
    'factual_score', 'section_coverage', 'graph_coherence', 'structure_signal'
]

systems = {
    'Baseline': dict(single_results['baseline_metrics']),
    'StructureAware': dict(single_results['structure_metrics']),
    'FactChecked': dict(single_results['fact_checked_metrics']),
}

# Final system = FactChecked + Phase2 media segmentation module
systems['FinalProposed'] = dict(systems['FactChecked'])
systems['FinalProposed']['phase2_media_score'] = phase2['proposed']['phase2_media_score']
systems['Baseline']['phase2_media_score'] = phase2['baseline']['phase2_media_score']
systems['StructureAware']['phase2_media_score'] = phase2['baseline']['phase2_media_score']
systems['FactChecked']['phase2_media_score'] = phase2['baseline']['phase2_media_score']

for name, data in systems.items():
    if 'runtime_sec' not in data or data.get('runtime_sec') == '-':
        data['runtime_sec'] = single_results['structure_metrics']['runtime_sec']

rows = []
for sys_name, data in systems.items():
    row = {'system': sys_name}
    for k in metric_keys + ['phase2_media_score', 'runtime_sec']:
        row[k] = float(data.get(k, 0) if data.get(k, 0) != '-' else 0)
    rows.append(row)

df_system = pd.DataFrame(rows)

# Composite score designed for publication reporting (explicit weighted criteria)
weights = {
    'rougeL_f1': 0.15,
    'semantic_f1_proxy': 0.25,
    'factual_score': 0.30,
    'section_coverage': 0.15,
    'structure_signal': 0.10,
    'phase2_media_score': 0.05,
}

df_system['composite_novelty_score'] = sum(df_system[m] * w for m, w in weights.items())

baseline_row = df_system[df_system['system'] == 'Baseline'].iloc[0]
for col in metric_keys + ['phase2_media_score', 'composite_novelty_score']:
    df_system[f'delta_vs_baseline_{col}'] = df_system[col] - baseline_row[col]

main_table_path = TAB_DIR / 'publication_main_metrics.csv'
delta_table_path = TAB_DIR / 'publication_delta_metrics.csv'
latex_table_path = TAB_DIR / 'publication_main_metrics.tex'

df_system.to_csv(main_table_path, index=False)

delta_cols = ['system'] + [f'delta_vs_baseline_{m}' for m in metric_keys + ['phase2_media_score', 'composite_novelty_score']]
df_system[delta_cols].to_csv(delta_table_path, index=False)

with open(latex_table_path, 'w', encoding='utf-8') as f:
    f.write(df_system[['system'] + metric_keys + ['phase2_media_score', 'composite_novelty_score', 'runtime_sec']].to_latex(index=False, float_format='%.4f'))

# Plot 1: grouped metric comparison
plot_metrics = ['rougeL_f1', 'semantic_f1_proxy', 'factual_score', 'section_coverage', 'structure_signal', 'phase2_media_score']
plot_df = df_system.set_index('system')[plot_metrics]
ax = plot_df.plot(kind='bar', figsize=(12, 6), width=0.8)
ax.set_title('System Comparison Across Core Metrics')
ax.set_ylabel('Score')
ax.set_xlabel('System')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plot1 = FIG_DIR / 'publication_metric_bar.png'
plt.savefig(plot1, dpi=300)
plt.close()

# Plot 2: composite score and runtime trade-off
fig, ax1 = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_system))
ax1.bar(x - 0.15, df_system['composite_novelty_score'], width=0.3, label='Composite novelty score')
ax1.set_ylabel('Composite novelty score')
ax1.set_xticks(x)
ax1.set_xticklabels(df_system['system'], rotation=15)

ax2 = ax1.twinx()
ax2.plot(x + 0.15, df_system['runtime_sec'], color='crimson', marker='o', linewidth=2, label='Runtime (sec)')
ax2.set_ylabel('Runtime (sec)', color='crimson')

ax1.set_title('Quality vs Runtime Trade-off')
fig.tight_layout()
plot2 = FIG_DIR / 'publication_quality_runtime_tradeoff.png'
plt.savefig(plot2, dpi=300)
plt.close()

# Plot 3: radar chart for publication metrics
radar_metrics = ['rougeL_f1', 'semantic_f1_proxy', 'factual_score', 'section_coverage', 'structure_signal', 'phase2_media_score']
radar_df = df_system[['system'] + radar_metrics].copy()
for m in radar_metrics:
    mn, mx = radar_df[m].min(), radar_df[m].max()
    radar_df[m] = 0.5 if abs(mx - mn) < 1e-9 else (radar_df[m] - mn) / (mx - mn)

angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(7, 7))
ax = plt.subplot(111, polar=True)
for _, row in radar_df.iterrows():
    values = row[radar_metrics].tolist() + [row[radar_metrics].tolist()[0]]
    ax.plot(angles, values, linewidth=2, label=row['system'])
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics)
ax.set_yticklabels([])
ax.set_title('Normalized Radar Comparison (Higher is Better)')
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15))
plot3 = FIG_DIR / 'publication_radar_comparison.png'
plt.tight_layout()
plt.savefig(plot3, dpi=300)
plt.close()

print('Saved tables:')
print('-', main_table_path)
print('-', delta_table_path)
print('-', latex_table_path)
print()
print('Saved plots:')
print('-', plot1)
print('-', plot2)
print('-', plot3)
print()
print('Composite scores:')
print(df_system[['system', 'composite_novelty_score', 'runtime_sec']])


Saved tables:
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/tables/publication_main_metrics.csv
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/tables/publication_delta_metrics.csv
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/tables/publication_main_metrics.tex

Saved plots:
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/figures/publication_metric_bar.png
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/figures/publication_quality_runtime_tradeoff.png
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/figures/publication_radar_comparison.png

Composite scores:
           system  composite_novelty_score  runtime_sec
0        Baseline                 0.311555      151.459
1  StructureAware                 0.441020      237.094
2     FactChecked                 0.441020      237.094
3   FinalProposed                 0.465395      237.094


## 3B. Relation Graphs (Section and Media)

In [5]:
# RELATION_GRAPHS
paper = papers[0]

# Graph A: section-to-section relation graph from structure-aware links
sec_graph = single_results['section_graph']
G = nx.DiGraph()
for src, edges in sec_graph.items():
    G.add_node(src)
    for dst, w in edges:
        if w >= 0.08:
            G.add_edge(src, dst, weight=w)

# keep graph readable for publication
if len(G.nodes) > 14:
    top_nodes = sorted(G.out_degree, key=lambda x: x[1], reverse=True)[:14]
    keep = {n for n, _ in top_nodes}
    G = G.subgraph(keep).copy()

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=0.9)
weights = [G[u][v]['weight'] for u, v in G.edges()]
nx.draw_networkx_nodes(G, pos, node_size=1000, node_color='#cfe8ff')
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edges(G, pos, arrows=True, width=[1 + 2*w for w in weights], alpha=0.7)
plt.title('Section Relation Graph (Structure-Aware Links)')
plt.axis('off')
plot_sec = FIG_DIR / 'publication_section_relation_graph.png'
plt.tight_layout()
plt.savefig(plot_sec, dpi=300)
plt.close()

# Graph B: media-to-section bipartite graph
media_items = phase2['proposed']['per_item']
B = nx.Graph()
for item in media_items:
    media_id = item['id']
    B.add_node(media_id, bipartite='media')
    for sec in item.get('assigned_sections', []):
        B.add_node(sec, bipartite='section')
        B.add_edge(media_id, sec)

media_nodes = [n for n, d in B.nodes(data=True) if d.get('bipartite') == 'media']
section_nodes = [n for n, d in B.nodes(data=True) if d.get('bipartite') == 'section']

plt.figure(figsize=(12, 7))
pos = {}
pos.update((n, (0, i)) for i, n in enumerate(media_nodes))
pos.update((n, (1, i)) for i, n in enumerate(section_nodes))
nx.draw_networkx_nodes(B, pos, nodelist=media_nodes, node_color='#ffd6a5', node_size=900, label='Media')
nx.draw_networkx_nodes(B, pos, nodelist=section_nodes, node_color='#bde0fe', node_size=1000, label='Sections')
nx.draw_networkx_edges(B, pos, alpha=0.7)
nx.draw_networkx_labels(B, pos, font_size=8)
plt.title('Media-Section Assignment Graph')
plt.axis('off')
plt.legend(loc='upper left')
plot_media = FIG_DIR / 'publication_media_section_graph.png'
plt.tight_layout()
plt.savefig(plot_media, dpi=300)
plt.close()

# Save adjacency/incidence tables for publication appendix
section_nodes_sorted = sorted(set(sec_graph.keys()))
adj = pd.DataFrame(0.0, index=section_nodes_sorted, columns=section_nodes_sorted)
for src, edges in sec_graph.items():
    for dst, w in edges:
        adj.loc[src, dst] = w
adj_path = TAB_DIR / 'section_relation_adjacency.csv'
adj.to_csv(adj_path)

all_sections = sorted(paper['sections'].keys())
all_media = [m['id'] for m in media_items]
inc = pd.DataFrame(0, index=all_sections, columns=all_media)
for m in media_items:
    for sec in m.get('assigned_sections', []):
        if sec in inc.index and m['id'] in inc.columns:
            inc.loc[sec, m['id']] = 1
inc_path = TAB_DIR / 'media_section_incidence.csv'
inc.to_csv(inc_path)

print('Saved relation plots:')
print('-', plot_sec)
print('-', plot_media)
print('Saved relation tables:')
print('-', adj_path)
print('-', inc_path)


Saved relation plots:
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/figures/publication_section_relation_graph.png
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/figures/publication_media_section_graph.png
Saved relation tables:
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/tables/section_relation_adjacency.csv
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/tables/media_section_incidence.csv


## 3C. Section-Level Ablation Analysis

In [6]:
# SECTION_ABLATION
paper = papers[0]
sections = paper['sections']
paper_title = paper['metadata'].get('title', 'Untitled')

ranked_items = sorted(
    sections.items(),
    key=lambda x: pipeline.structure.section_importance(x[0], x[1].get('content', '')),
    reverse=True,
)
section_subset = dict(ranked_items[:6])

baseline_sec = pipeline.structure.summarize_independent(section_subset, paper_title)
structure_sec = pipeline.structure.summarize_structure_aware(section_subset, paper_title)

rows = []
for sec_title, sec in section_subset.items():
    source = sec.get('content', '')
    reference = ' '.join(source.split()[:300])
    b = pipeline.evaluator.evaluate_summary(baseline_sec.get(sec_title, ''), reference, source)
    s = pipeline.evaluator.evaluate_summary(structure_sec.get(sec_title, ''), reference, source)
    rows.append({
        'section': sec_title,
        'baseline_rougeL': b['rougeL_f1'],
        'structure_rougeL': s['rougeL_f1'],
        'delta_rougeL': s['rougeL_f1'] - b['rougeL_f1'],
        'baseline_semantic': b['semantic_f1_proxy'],
        'structure_semantic': s['semantic_f1_proxy'],
        'delta_semantic': s['semantic_f1_proxy'] - b['semantic_f1_proxy'],
        'baseline_factual': b['factual_score'],
        'structure_factual': s['factual_score'],
        'delta_factual': s['factual_score'] - b['factual_score'],
    })

df_ablation = pd.DataFrame(rows).sort_values('delta_semantic', ascending=False)
ablation_table_path = TAB_DIR / 'section_level_ablation.csv'
df_ablation.to_csv(ablation_table_path, index=False)

# Plot: per-section metric deltas heatmap-style
heat_cols = ['delta_rougeL', 'delta_semantic', 'delta_factual']
heat_data = df_ablation.set_index('section')[heat_cols]

fig, ax = plt.subplots(figsize=(10, max(4, 0.7 * len(heat_data))))
im = ax.imshow(heat_data.values, cmap='coolwarm', aspect='auto')
ax.set_xticks(np.arange(len(heat_cols)))
ax.set_xticklabels(heat_cols)
ax.set_yticks(np.arange(len(heat_data.index)))
ax.set_yticklabels(heat_data.index)
for i in range(len(heat_data.index)):
    for j in range(len(heat_cols)):
        ax.text(j, i, f"{heat_data.values[i, j]:.3f}", ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
ax.set_title('Section-Level Delta: StructureAware - Baseline')
plt.tight_layout()
plot_delta = FIG_DIR / 'section_level_delta_heatmap.png'
plt.savefig(plot_delta, dpi=300)
plt.close()

# Plot: win count by metric
win_counts = {
    'ROUGE-L wins': int((df_ablation['delta_rougeL'] > 0).sum()),
    'Semantic wins': int((df_ablation['delta_semantic'] > 0).sum()),
    'Factual wins': int((df_ablation['delta_factual'] > 0).sum()),
}

plt.figure(figsize=(7, 4))
plt.bar(win_counts.keys(), win_counts.values(), color=['#8ecae6', '#219ebc', '#023047'])
plt.ylabel('No. of sections with improvement')
plt.title('Section-Level Improvement Count (StructureAware vs Baseline)')
for i, v in enumerate(win_counts.values()):
    plt.text(i, v + 0.05, str(v), ha='center', fontsize=10)
plt.tight_layout()
plot_win = FIG_DIR / 'section_level_win_count.png'
plt.savefig(plot_win, dpi=300)
plt.close()

print('Saved section ablation table:', ablation_table_path)
print('Saved section ablation plots:')
print('-', plot_delta)
print('-', plot_win)
print()
print('Section-level deltas preview:')
print(df_ablation[['section', 'delta_rougeL', 'delta_semantic', 'delta_factual']])


Saved section ablation table: /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/tables/section_level_ablation.csv
Saved section ablation plots:
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/figures/section_level_delta_heatmap.png
- /home/cdac/Office-Projects/Research-Paper-Summarizer/outputs/figures/section_level_win_count.png

Section-level deltas preview:
                            section  ...  delta_factual
4        Task speciﬁc model details  ...        -0.0261
3  Autoregressive Language Modeling  ...         0.1388
5                          Abstract  ...        -0.0156
0                       Results (2)  ...        -0.0764
2                      Introduction  ...        -0.0976
1                           Results  ...        -0.1365

[6 rows x 4 columns]


## 4. IDEA 3: Multi-document research summarization

In [7]:
multi_input = papers if len(papers) > 1 else papers * 3
multi_results = pipeline.run_multi_document_experiment(multi_input)
print('Combined summary:\n', multi_results['combined_summary'])
print('\nCommon findings:', multi_results['common_findings'])
print('\nDifferences:')
for d in multi_results['differences']:
    print('-', d)
print('\nTrends:', multi_results['trends'])


Combined summary:
 The Longformer model is a significant improvement over traditional transformer-based models in long-document tasks. It achieves state-of-the-art results on text8 and enwik8 datasets with a small BPC of 1.10 and 1.00 respectively, outperforming comparable sparse transformer models. The unique attention mechanism scales linearly with sequence length, making it suitable for long documents. The Longformer-Encoder-Decoder variant demonstrates further effectiveness, and its results match or exceed prior work. The paper concludes that Longformer is a viable solution for long document tasks.

Common findings: ['attention', 'al', 'et', 'longformer', 'model', '1', '2019', '2020', 'roberta', 'sequence', 'long', '2']

Differences:
- Longformer: The Long-Document Transformer: The Longformer model is a significant improvement over traditional transformer-based models, particularly in the context of long-document tasks. It achieves state-of-the-art results on text8 and enwik8 datas

## 5. IDEA 5 + IDEA 6: Domain-specific + Interactive assistant

In [8]:
print('Detected domain:', single_results['detected_domain'])
print('\nDomain-specific summary:\n', single_results['domain_summary'])
print('\nQ/A:\n', single_results['qa_answer'])
print('\nContributions:\n', single_results['contributions'])
print('\nLimitations:\n', single_results['limitations'])
print('\nFuture work:\n', single_results['future_work'])


Detected domain: legal

Domain-specific summary:
 The authors propose a Longformer model, which addresses the scalability limitation of self-attention in transformer-based architectures. By introducing a linearly scaled attention mechanism, the model can process long sequences more efficiently, achieving state-of-the-art results on various character-level language modeling tasks and long document tasks. The authors also pretrain and fine-tune the model on a variety of downstream tasks, demonstrating its effectiveness on downstream tasks. Finally, the authors introduce the Longformer-Encoder-Decoder (LED) variant for long document generative sequence-to-sequence tasks, achieving state-of-the-art results on the arXiv sum- marization dataset. Overall, the proposed Longformer model and its variants outperform previous state-of-the-art models on long document tasks.

Q/A:
 This paper presents the ETC (Encoding long and structured inputs in transformers) model, which uses a new dilated atten

## 6. Human Evaluation Template

In [9]:
for row in single_results['human_eval_template']:
    print(row)


{'criterion': 'Informativeness', 'baseline_score_1_to_5': '', 'proposed_score_1_to_5': '', 'notes': ''}
{'criterion': 'Factual consistency', 'baseline_score_1_to_5': '', 'proposed_score_1_to_5': '', 'notes': ''}
{'criterion': 'Section coverage', 'baseline_score_1_to_5': '', 'proposed_score_1_to_5': '', 'notes': ''}
{'criterion': 'Readability', 'baseline_score_1_to_5': '', 'proposed_score_1_to_5': '', 'notes': ''}


## 7. Save consolidated results

In [10]:
payload = {
    'single': single_results,
    'phase2_media': phase2,
    'multi': multi_results,
    'metric_table': pretty_metric_table(single_results),
}
out = ROOT / 'research_experiment_results.json'
out.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print('Saved:', out)


Saved: /home/cdac/Office-Projects/Research-Paper-Summarizer/research_experiment_results.json


## Suggested Paper Title
**Structure-Aware and Fact-Consistent Summarization of Scientific Documents using LLMs with Media-Aware Phase-2 Segmentation**